# TP3 – Exploration des données

Ryma Dinari – Retail Intelligence Platform

Dans ce notebook je fais les étapes 1 à 3 de l'UC-06 : je vérifie mon environnement, je fais l'inventaire des sources et j'explore les données avant de les nettoyer.

J'ai choisi 4 fichiers du kit TPCx-BB (le même que pour les jours 1 et 2) :
- `store_sales` : les ventes en magasin
- `item` : les produits
- `store` : les magasins
- `date_dim` : le calendrier

Ce sont ceux dont j'ai besoin pour mes questions Q1, Q3 et Q5 du jour 1 (évolution du CA, meilleures catégories, meilleurs magasins).

## 1. Environnement

In [1]:
import sys
sys.path.insert(0, "..")

from config import *
import pyspark
from pyspark.sql import functions as F

spark = creer_session()

print("Python  :", sys.version.split()[0])
print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)
print("Mode    :", spark.sparkContext.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 13:53:27 WARN Utils: Your hostname, rymadinari-VivoBook-ASUSLaptop-M1702QA-M1702QA, resolves to a loopback address: 127.0.1.1; using 10.188.47.136 instead (on interface wlp1s0)
26/11/18 13:53:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 13:53:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/18 13:53:28 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Python  : 3.12.3
PySpark : 4.3.0
Spark   : 4.3.0
Mode    : local[*]


In [2]:
# petit test d'écriture / lecture dans data/test
df_test = spark.createDataFrame([("Magasin A", 1200.0), ("Magasin B", 850.0)], ["magasin", "ca"])
df_test.write.mode("overwrite").parquet(str(TEST / "test"))
spark.read.parquet(str(TEST / "test")).show()

+---------+------+
|  magasin|    ca|
+---------+------+
|Magasin A|1200.0|
|Magasin B| 850.0|
+---------+------+



Spark 4.3.0 fonctionne en local avec Java 17 (installé au jour 2). L'écriture et la lecture marchent.
Je travaille sur mon PC, pas sur un cluster : je lis les fichiers depuis le dossier local du kit.

## 2. Inventaire des sources

Les fichiers n'ont pas d'en-tête et le séparateur est `|`. Je lis tout en texte avec la fonction `lire_brut` (dans `config.py`), et je donne les noms de colonnes moi-même. Je n'utilise pas `inferSchema`, je préfère vérifier les types moi-même.

In [3]:
tables = ["store_sales", "item", "store", "date_dim"]
bruts = {t: lire_brut(spark, t) for t in tables}

for t in tables:
    df = bruts[t]
    taille = round(chemin_source(t).stat().st_size / 1024)
    print(f"{t:12} {taille:>6} Ko   {df.count():>7} lignes   {len(df.columns)} colonnes")

store_sales   90591 Ko    667579 lignes   23 colonnes


item           6640 Ko     17820 lignes   22 colonnes
store             3 Ko        12 lignes   29 colonnes


date_dim      15035 Ko    109573 lignes   28 colonnes


On retrouve les mêmes nombres de lignes qu'aux jours 1 et 2, donc la lecture est bonne.

## 3. Exploration

### store_sales

In [4]:
ventes = bruts["store_sales"]
ventes.printSchema()
ventes.show(5)

root
 |-- ss_sold_date_sk: string (nullable = true)
 |-- ss_sold_time_sk: string (nullable = true)
 |-- ss_item_sk: string (nullable = true)
 |-- ss_customer_sk: string (nullable = true)
 |-- ss_cdemo_sk: string (nullable = true)
 |-- ss_hdemo_sk: string (nullable = true)
 |-- ss_addr_sk: string (nullable = true)
 |-- ss_store_sk: string (nullable = true)
 |-- ss_promo_sk: string (nullable = true)
 |-- ss_ticket_number: string (nullable = true)
 |-- ss_quantity: string (nullable = true)
 |-- ss_wholesale_cost: string (nullable = true)
 |-- ss_list_price: string (nullable = true)
 |-- ss_sales_price: string (nullable = true)
 |-- ss_ext_discount_amt: string (nullable = true)
 |-- ss_ext_sales_price: string (nullable = true)
 |-- ss_ext_wholesale_cost: string (nullable = true)
 |-- ss_ext_list_price: string (nullable = true)
 |-- ss_ext_tax: string (nullable = true)
 |-- ss_coupon_amt: string (nullable = true)
 |-- ss_net_paid: string (nullable = true)
 |-- ss_net_paid_inc_tax: string (n

In [5]:
# valeurs nulles par colonne
for c in ventes.columns:
    nb = ventes.filter(F.col(c).isNull()).count()
    if nb > 0:
        print(c, ":", nb)

ss_cdemo_sk : 238


ss_hdemo_sk : 295


ss_addr_sk : 448


ss_promo_sk : 158


Les seules colonnes avec des vides sont des clés facultatives (promotion, adresse, infos démographiques du client). Les colonnes importantes (date, produit, magasin, ticket, montants) sont toujours remplies.

In [6]:
# est-ce que les valeurs peuvent bien être converties en nombres ?
# try_cast renvoie null quand la conversion ne marche pas
for c, type_cible in COLONNES["store_sales"]:
    nb = ventes.filter(F.col(c).isNotNull() & F.col(c).try_cast(type_cible).isNull()).count()
    if nb > 0:
        print(c, ":", nb, "valeurs non convertibles")
print("vérification terminée")

vérification terminée


Aucune valeur bloquante, donc les types que j'ai choisis (entiers pour les clés, décimaux pour les montants) sont bons.

Doublons : une vente doit normalement être unique pour un ticket + un produit.

In [7]:
total = ventes.count()
print("lignes totalement identiques :", total - ventes.distinct().count())
print("doublons ticket + produit    :", total - ventes.select("ss_ticket_number", "ss_item_sk").distinct().count())

lignes totalement identiques : 0


doublons ticket + produit    : 4136


In [8]:
# je regarde quelques exemples
from pyspark.sql import Window

w = Window.partitionBy("ss_ticket_number", "ss_item_sk")
(ventes.withColumn("nb", F.count("*").over(w))
       .filter("nb > 1")
       .select("ss_ticket_number", "ss_item_sk", "ss_sold_date_sk", "ss_store_sk",
               "ss_customer_sk", "ss_quantity", "ss_sales_price")
       .orderBy("ss_ticket_number", "ss_item_sk")
       .show(6))

+----------------+----------+---------------+-----------+--------------+-----------+--------------+
|ss_ticket_number|ss_item_sk|ss_sold_date_sk|ss_store_sk|ss_customer_sk|ss_quantity|ss_sales_price|
+----------------+----------+---------------+-----------+--------------+-----------+--------------+
|           10030|     10611|          37093|          4|         24315|         89|          2.96|
|           10030|     10611|          37093|          4|         24315|         88|         12.67|
|           10062|     10001|          37094|          3|         31030|         88|         15.48|
|           10062|     10001|          37094|          3|         31030|         73|          0.00|
|             101|     10595|          36892|          6|         67007|         78|          1.54|
|             101|     10595|          36892|          6|         67007|         36|         12.71|
+----------------+----------+---------------+-----------+--------------+-----------+--------------+


Il n'y a pas de lignes copiées. Les « doublons » ont le même ticket, le même jour et le même client, mais pas la même quantité ni le même prix : c'est le même produit passé deux fois sur le ticket. Je les garde.

Valeurs bizarres sur les montants :

In [9]:
v = ventes.select(
    F.col("ss_quantity").cast("int").alias("quantite"),
    F.col("ss_list_price").cast("decimal(7,2)").alias("prix_catalogue"),
    F.col("ss_sales_price").cast("decimal(7,2)").alias("prix_vente"),
    F.col("ss_net_paid").cast("decimal(7,2)").alias("net_paye"),
    F.col("ss_net_profit").cast("decimal(7,2)").alias("marge"),
)
v.describe().show()

print("prix de vente > prix catalogue :", v.filter("prix_vente > prix_catalogue").count())
print("prix de vente à 0              :", v.filter("prix_vente = 0").count())
print("marge négative                 :", v.filter("marge < 0").count())

+-------+------------------+------------------+-----------------+-----------------+------------------+
|summary|          quantite|    prix_catalogue|       prix_vente|         net_paye|             marge|
+-------+------------------+------------------+-----------------+-----------------+------------------+
|  count|            667579|            667579|           667579|           667579|            667579|
|   mean|50.564014146640325|         75.725878|        37.863697|      1723.435858|       -827.697358|
| stddev|28.880895668592885|46.007204487184474|34.39915676234274|2167.898616478136|1727.4881593984771|
|    min|                 1|              1.00|             0.00|             0.00|          -9866.60|
|    max|               100|            200.00|           199.30|         18745.00|           9134.00|
+-------+------------------+------------------+-----------------+-----------------+------------------+



prix de vente > prix catalogue : 0


prix de vente à 0              : 3324


marge négative                 : 495971


- Quantités entre 1 et 100, pas de montant négatif, pas de prix au-dessus du prix catalogue.
- 3 324 ventes à 0 € : ça peut être un produit offert, je les garde.
- Environ 74 % des ventes ont une marge négative (déjà vu au jour 1). Les calculs dans le fichier sont cohérents, c'est le générateur de données qui fait ça. Je garde les lignes mais je ne vais pas trop interpréter la marge.

### item

In [10]:
produits = bruts["item"]
produits.select("i_item_sk", "i_item_id", "i_current_price", "i_category", "i_brand").show(5)

for c in produits.columns:
    nb = produits.filter(F.col(c).isNull()).count()
    if nb > 0:
        print(c, ":", nb)

print("doublons i_item_sk :", produits.count() - produits.select("i_item_sk").distinct().count())

+---------+----------------+---------------+-----------------+--------------------+
|i_item_sk|       i_item_id|i_current_price|       i_category|             i_brand|
+---------+----------------+---------------+-----------------+--------------------+
|        0|AAAAAAAAAAAAAAAA|          72.29|Sports & Outdoors|71TS7NSbvH1YbdiQM...|
|        1|AAAAAAAAAAAAAAAB|          95.67|            Books|     qNBimHGBu763udH|
|        2|AAAAAAAAAAAAAAAC|          62.74|            Books|                 ma5|
|        3|AAAAAAAAAAAAAAAD|          98.67|            Books|   op9d25ZWK4zQ7JNAc|
|        4|AAAAAAAAAAAAAAAE|          40.95|            Books| RYLn4HY3I3kooPCNhwV|
+---------+----------------+---------------+-----------------+--------------------+
only showing top 5 rows


i_rec_end_date : 17820
i_item_desc : 4


i_brand_id : 8
i_brand : 6


doublons i_item_sk : 0


In [11]:
produits.groupBy("i_category").count().orderBy(F.desc("count")).show()

+--------------------+-----+
|          i_category|count|
+--------------------+-----+
|               Music| 8143|
|               Books| 5335|
|         Movies & TV| 1286|
|         Electronics|  692|
|Clothing & Access...|  610|
|      Home & Kitchen|  476|
|   Sports & Outdoors|  465|
|Tools & Home Impr...|  408|
|        Toys & Games|  405|
+--------------------+-----+



La clé `i_item_sk` est unique et la catégorie est toujours remplie (9 catégories). Il manque juste quelques descriptions et marques, et `i_rec_end_date` est vide partout (normal, chaque produit n'a qu'une version).

### store

In [12]:
magasins = bruts["store"]
print("doublons s_store_sk :", magasins.count() - magasins.select("s_store_sk").distinct().count())
magasins.select("s_store_sk", "s_store_name", "s_rec_start_date", "s_closed_date_sk",
                "s_number_employees", "s_floor_space", "s_state").show()

doublons s_store_sk : 0
+----------+------------+----------------+----------------+------------------+-------------+-------+
|s_store_sk|s_store_name|s_rec_start_date|s_closed_date_sk|s_number_employees|s_floor_space|s_state|
+----------+------------+----------------+----------------+------------------+-------------+-------+
|         0|      Forest|      2000-08-02|           71238|               217|      8891512|     CO|
|         1|    Hillside|      2000-07-03|           18994|               244|      8751339|     TN|
|         2|    Sherwood|      2000-09-01|            8900|               266|      5913334|     CA|
|         3|     Pioneer|      2000-06-02|           69902|               221|      5777026|     TX|
|         4|     Elmwood|      2000-07-03|           79617|               254|      7508523|     KY|
|         5|       Egypt|      2000-03-03|           88139|               215|      6942447|     KY|
|         6|  Bloomfield|      2000-01-03|           15761|        

In [13]:
# s_closed_date_sk est une clé de date : je regarde à quelles dates ça correspond
fermetures = [r.s_closed_date_sk for r in magasins.collect()]
bruts["date_dim"].filter(F.col("d_date_sk").isin(fermetures)).select("d_date").orderBy("d_date").show()

+----------+
|    d_date|
+----------+
|1917-01-27|
|1924-05-15|
|1943-02-26|
|1952-01-03|
|1968-10-22|
|2019-06-20|
|2038-10-13|
|2091-05-21|
|2095-01-16|
|2095-08-12|
|2117-12-26|
|2141-04-26|
+----------+



12 magasins. Les dates de fermeture vont de 1917 à 2141 alors que les magasins ouvrent en 2000 et vendent jusqu'en 2005, donc cette colonne n'a pas de sens. Les surfaces (plusieurs millions) ne sont pas réalistes non plus.

### date_dim

In [14]:
dates = bruts["date_dim"]
print("doublons d_date_sk :", dates.count() - dates.select("d_date_sk").distinct().count())
dates.select(F.min("d_date"), F.max("d_date")).show()
dates.groupBy("d_holiday").count().show()

doublons d_date_sk : 0
+-----------+-----------+
|min(d_date)|max(d_date)|
+-----------+-----------+
| 1900-01-01| 2199-12-31|
+-----------+-----------+



+---------+------+
|d_holiday| count|
+---------+------+
|        Y|109573|
+---------+------+



Calendrier de 1900 à 2199, sans trou ni doublon. Par contre `d_holiday` vaut Y partout, je ne l'utiliserai pas.

### Relations entre les tables

In [15]:
# ventes qui n'ont pas de correspondance dans la table de référence
liens = [("ss_item_sk", "item", "i_item_sk"),
         ("ss_store_sk", "store", "s_store_sk"),
         ("ss_sold_date_sk", "date_dim", "d_date_sk")]

for cle, table, cle_ref in liens:
    ref = bruts[table]
    nb = ventes.join(ref, ventes[cle] == ref[cle_ref], "left_anti").count()
    print(cle, "->", table, ":", nb, "orphelin(s)")

ss_item_sk -> item : 0 orphelin(s)


ss_store_sk -> store : 0 orphelin(s)


ss_sold_date_sk -> date_dim : 0 orphelin(s)


In [16]:
# période couverte par les ventes
ventes.join(dates, ventes.ss_sold_date_sk == dates.d_date_sk) \
      .select(F.min("d_date").alias("debut"), F.max("d_date").alias("fin")).show()

+----------+----------+
|     debut|       fin|
+----------+----------+
|2001-01-01|2005-12-13|
+----------+----------+



Aucune vente orpheline, donc je peux faire les jointures ventes → produits, magasins et dates sans perdre de lignes.
Les ventes vont du 01/01/2001 au 13/12/2005 : l'année 2005 n'est pas complète, il faudra en tenir compte.

## Ce que je retiens pour le nettoyage

- les types sont bons, je peux les appliquer directement ;
- les vides sont seulement sur des clés facultatives ;
- les doublons ticket + produit sont de vraies ventes, je les garde ;
- je garde les marges négatives et les ventes à 0 € ;
- je n'utilise pas `s_closed_date_sk` ni `d_holiday` ;
- 2005 est incomplète.